In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Suapan balik klasik dan aliran kawalan (litar dinamik)

<Accordion>
<AccordionItem title="Versi pakej">

Kod pada halaman ini dibangunkan menggunakan keperluan berikut.
Kami mengesyorkan penggunaan versi ini atau yang lebih baharu.

```
qiskit[all]~=2.4.0
```
</AccordionItem>
</Accordion>
Litar dinamik ialah alat yang berkuasa yang membolehkan anda mengukur Qubit di tengah-tengah pelaksanaan Circuit kuantum dan kemudian melakukan operasi logik klasik dalam Circuit, berdasarkan keputusan pengukuran mid-Circuit tersebut. Proses ini juga dikenali sebagai _suapan balik klasik_. Walaupun ini masih peringkat awal dalam memahami cara terbaik memanfaatkan litar dinamik, komuniti penyelidikan kuantum telah mengenal pasti beberapa kes penggunaan, seperti berikut:

* Penyediaan keadaan kuantum yang cekap, seperti [keadaan GHZ](https://journals.aps.org/prxquantum/abstract/10.1103/PRXQuantum.5.030339), [keadaan-W](https://arxiv.org/abs/2403.07604) (untuk maklumat lanjut tentang keadaan-W, rujuk juga ["State preparation by shallow circuits using feed forward"](https://arxiv.org/abs/2307.14840)), dan kelas luas [matrix product states](https://arxiv.org/abs/2404.16083)
* [Keterikatan jarak jauh yang cekap](https://journals.aps.org/prxquantum/abstract/10.1103/PRXQuantum.5.030339) antara Qubit pada cip yang sama dengan menggunakan Circuit cetek
* [Pensampelan cekap litar seperti IQP](https://arxiv.org/pdf/2505.04705)
## Pernyataan `if`
Pernyataan `if` digunakan untuk melaksanakan operasi secara bersyarat berdasarkan nilai bit atau daftar klasik.

Dalam contoh di bawah, kami menggunakan Gate Hadamard pada sebuah Qubit dan mengukurnya. Jika hasilnya ialah 1, maka kami menggunakan Gate X pada Qubit tersebut, yang berkesan membalikkannya semula ke keadaan 0. Kemudian kami mengukur Qubit itu sekali lagi. Hasil pengukuran seharusnya ialah 0 dengan kebarangkalian 100%.

In [1]:
from qiskit.circuit import QuantumCircuit, QuantumRegister, ClassicalRegister

qubits = QuantumRegister(1)
clbits = ClassicalRegister(1)
circuit = QuantumCircuit(qubits, clbits)
(q0,) = qubits
(c0,) = clbits

circuit.h(q0)
circuit.measure(q0, c0)
with circuit.if_test((c0, 1)):
    circuit.x(q0)
circuit.measure(q0, c0)
circuit.draw("mpl")

# example output counts: {'0': 1024}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/60924bfa-50ed-4d9d-a17b-9d64f2cc053f-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/60924bfa-50ed-4d9d-a17b-9d64f2cc053f-0.svg)

Pernyataan `with` boleh diberi sasaran tugasan yang itu sendiri merupakan pengurus konteks yang boleh disimpan dan kemudiannya digunakan untuk mencipta blok else, yang dilaksanakan apabila kandungan blok `if` *tidak* dilaksanakan.

Dalam contoh di bawah, kami memulakan daftar dengan dua Qubit dan dua bit klasik. Kami menggunakan Gate Hadamard pada Qubit pertama dan mengukurnya. Jika hasilnya ialah 1, maka kami menggunakan Gate Hadamard pada Qubit kedua; jika tidak, kami menggunakan Gate X pada Qubit kedua. Akhirnya, kami mengukur Qubit kedua juga.

In [2]:
qubits = QuantumRegister(2)
clbits = ClassicalRegister(2)
circuit = QuantumCircuit(qubits, clbits)
(q0, q1) = qubits
(c0, c1) = clbits

circuit.h(q0)
circuit.measure(q0, c0)
with circuit.if_test((c0, 1)) as else_:
    circuit.h(q1)
with else_:
    circuit.x(q1)
circuit.measure(q1, c1)

circuit.draw("mpl")

# example output counts: {'01': 260, '11': 272, '10': 492}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/20f0640a-a3f7-41b3-aada-b66bc89b0555-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/20f0640a-a3f7-41b3-aada-b66bc89b0555-0.svg)

Selain daripada bersyarat pada satu bit klasik, adalah juga boleh bersyarat pada nilai daftar klasik yang terdiri daripada beberapa bit.

Dalam contoh di bawah, kami menggunakan Gate Hadamard pada dua Qubit dan mengukurnya. Jika hasilnya ialah `01`, iaitu Qubit pertama ialah 1 dan Qubit kedua ialah 0, maka kami menggunakan Gate X pada Qubit ketiga. Akhirnya, kami mengukur Qubit ketiga. Perhatikan bahawa untuk kejelasan, kami memilih untuk menentukan keadaan bit klasik ketiga, iaitu 0, dalam syarat `if`. Dalam lukisan Circuit, syarat ditunjukkan oleh bulatan pada bit klasik yang dijadikan syarat. Bulatan pepejal menunjukkan syarat pada 1, manakala bulatan bergarisan luar menunjukkan syarat pada 0.

In [3]:
qubits = QuantumRegister(3)
clbits = ClassicalRegister(3)
circuit = QuantumCircuit(qubits, clbits)
(q0, q1, q2) = qubits
(c0, c1, c2) = clbits

circuit.h([q0, q1])
circuit.measure(q0, c0)
circuit.measure(q1, c1)
with circuit.if_test((clbits, 0b001)):
    circuit.x(q2)
circuit.measure(q2, c2)

circuit.draw("mpl")

# example output counts: {'101': 269, '011': 260, '000': 252, '010': 243}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/98e8f552-4169-42a3-8182-e14e9ffb59e2-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/98e8f552-4169-42a3-8182-e14e9ffb59e2-0.svg)

## Ungkapan klasik
Modul ungkapan klasik Qiskit [`qiskit.circuit.classical`](https://docs.quantum.ibm.com/api/qiskit/circuit_classical) mengandungi representasi eksperimentasi operasi masa larian pada nilai klasik semasa pelaksanaan Circuit. Disebabkan had perkakasan, hanya syarat `QuantumCircuit.if_test()` yang disokong pada masa ini.

Contoh berikut menunjukkan bahawa anda boleh menggunakan pengiraan pariti untuk mencipta keadaan GHZ n-Qubit menggunakan litar dinamik. Pertama, jana $n/2$ pasangan Bell pada Qubit bersebelahan. Kemudian, sambungkan pasangan-pasangan ini menggunakan lapisan Gate CNOT di antara pasangan. Anda kemudian mengukur Qubit sasaran semua Gate CNOT sebelumnya dan menetapkan semula setiap Qubit yang diukur ke keadaan $\vert 0 \rangle$. Anda menggunakan $X$ pada setiap tapak yang tidak diukur di mana pariti semua bit sebelumnya adalah ganjil. Akhirnya, Gate CNOT digunakan pada Qubit yang diukur untuk memulihkan keterikatan yang hilang dalam pengukuran.

Dalam pengiraan pariti, elemen pertama ungkapan yang dibina melibatkan pengangkatan objek Python `mr[0]` ke nod [`Value`](https://docs.quantum.ibm.com/api/qiskit/circuit_classical#value) (`lift` digunakan untuk menukar objek arbitrari kepada ungkapan klasik). Ini tidak diperlukan untuk `mr[1]` dan daftar klasik yang mungkin berikutnya, kerana ia adalah input kepada `expr.bit_xor`, dan sebarang pengangkatan yang diperlukan dilakukan secara automatik dalam kes-kes ini. Ungkapan sedemikian boleh dibina dalam gelung dan konstruk lain.

In [4]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.classical import expr

num_qubits = 8
if num_qubits % 2 or num_qubits < 4:
    raise ValueError("num_qubits must be an even integer ≥ 4")
meas_qubits = list(range(2, num_qubits, 2))  # qubits to measure and reset

qr = QuantumRegister(num_qubits, "qr")
mr = ClassicalRegister(len(meas_qubits), "m")
qc = QuantumCircuit(qr, mr)

# Create local Bell pairs
qc.reset(qr)
qc.h(qr[::2])
for ctrl in range(0, num_qubits, 2):
    qc.cx(qr[ctrl], qr[ctrl + 1])

# Glue neighboring pairs
for ctrl in range(1, num_qubits - 1, 2):
    qc.cx(qr[ctrl], qr[ctrl + 1])

# Measure boundary qubits between pairs,reset to 0
for k, q in enumerate(meas_qubits):
    qc.measure(qr[q], mr[k])
    qc.reset(qr[q])

# Parity-conditioned X corrections
# Each non-measured qubit gets flipped iff the parity (XOR) of all
# preceding measurement bits is 1
for tgt in range(num_qubits):
    if tgt in meas_qubits:  # skip measured qubits
        continue
    # all measurement registers whose physical qubit index < tgt
    left_bits = [k for k, q in enumerate(meas_qubits) if q < tgt]
    if not left_bits:  # skip if list empty
        continue

    # build XOR-parity expression
    parity = expr.lift(
        mr[left_bits[0]]
    )  # lift the first bit to Value so it will be treated like a boolean.
    for k in left_bits[1:]:
        parity = expr.bit_xor(
            mr[k], parity
        )  # calculate parity with all other bits
    with qc.if_test(parity):  # Add X if parity is 1
        qc.x(qr[tgt])

# Re-entangle measured qubits
for ctrl in range(1, num_qubits - 1, 2):
    qc.cx(qr[ctrl], qr[ctrl + 1])

In [5]:
qc.draw(output="mpl", style="iqp", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/d0f0abdb-50d5-408d-a704-a1a555acdd85-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/d0f0abdb-50d5-408d-a704-a1a555acdd85-0.svg)

## Langkah seterusnya
> **Tip:** - Ketahui cara melaksanakan dynamic decoupling yang tepat dengan menggunakan [stretch](/guides/stretch).
> - Guna [visualisasi jadual Circuit](/guides/qiskit-runtime-circuit-timing) untuk debug dan optimumkan litar dinamik anda.
> - [Laksanakan litar dinamik](/guides/execute-dynamic-circuits).